In [ ]:
import os
import warnings
import pickle    # chunk, vectorDB 저장한것 사용
from dotenv import load_dotenv

# 경고메세지 삭제
warnings.filterwarnings('ignore')
load_dotenv()

# openapi key 확인
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('.env확인,  key없음')

import numpy as np
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from time import time
from rank_bm25 import BM25Okapi
from typing import List

# HuggingFace 임베딩 import 시도
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except:
    print('pip install langchain-huggingface sentence-transformers')

# 임베딩 모델 정의
class KoreanEmbeddingModels:
    '''한국어 임베딩 모델 팩토리 클래스
    다양한 임베딩 모델을 쉽게 교체할 수 있도록 추상화'''

    @staticmethod  # 객체단위로 사용할 필요가 없는경우...????
    def get_bge_m3 (device: str='cpu'):
        '''BGE-M3 모델을 반환
        - Dense + Sparse 임베딩을 지원
        - 다국어 지원(한국어 우수)
        '''
        return HuggingFaceEmbeddings(
            model_name = "BAAI/bge-m3",
            model_kwargs = {
                'device':device,
                'trust_remote_code': True
            },
            encode_kwargs = {
                'normalize_embeddings':True,  # 정규화를 해서 코사인 유사도 계산을 용이하게 함
                'batch_size':32
            }
        )
    
    @staticmethod
    def get_multilingual_e5(device:str = 'cpu'):
        '''Multilingual-E5 모델 반환
        - 경량화
        - 다국어 지원
        '''
        return HuggingFaceEmbeddings(
            model_name = 'intfloat/multilingual-e5-large',
            model_kwargs = {'device': device},
            encode_kwargs = {'normalize_embeddings': True}
        )
    @staticmethod
    def get_korean_roberta(device:str = 'cpu'):
        '''BM-K/KoSimCSE-roberta-multitask'''
        return HuggingFaceEmbeddings(
            model_name = 'BM-K/KoSimCSE-roberta-multitask',
            model_kwargs = {'device': device},
            encode_kwargs = {'normalize_embeddings': True}
            )
    @staticmethod
    def get_openai(model:str='text-embedding-3-small'):
        '''OpenAI 임베딩 모델'''
        return OpenAIEmbeddings(model=model)


# 데이터 로드
korean_documents = [
    Document(
        page_content="""
        인공지능(AI)은 기계가 인간의 지능을 모방하여 학습하고, 추론하며, 
        문제를 해결할 수 있도록 하는 기술입니다. 최근 대규모 언어 모델(LLM)의 
        발전으로 AI는 자연어 처리, 번역, 요약 등 다양한 분야에서 활용되고 있습니다.
        특히 GPT-4, Claude, Gemini 등의 모델이 주목받고 있습니다.
        """,
        metadata={"source": "ai_intro", "topic": "인공지능"}
    ),
    Document(
        page_content="""
        RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술로,
        LLM의 한계를 보완합니다. 기업의 내부 문서나 최신 정보를 벡터 
        데이터베이스에 저장하고, 사용자 질문과 관련된 문서를 검색하여
        답변의 정확성을 높입니다. 이를 통해 환각(Hallucination) 현상을 
        줄일 수 있습니다.
        """,
        metadata={"source": "rag_intro", "topic": "RAG"}
    ),
    Document(
        page_content="""
        LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
        프롬프트 관리, 체인 구성, 메모리 시스템, 에이전트 등
        다양한 기능을 제공합니다. Python과 JavaScript 버전이 있으며,
        OpenAI, Anthropic, Hugging Face 등 다양한 모델과 통합됩니다.
        """,
        metadata={"source": "langchain_intro", "topic": "프레임워크"}
    ),
    Document(
        page_content="""
        벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는
        데이터베이스입니다. 텍스트, 이미지, 오디오 등을 임베딩 벡터로 
        변환하여 저장하면, 의미적으로 유사한 항목을 빠르게 찾을 수 있습니다.
        ChromaDB, Pinecone, Weaviate, FAISS 등이 대표적입니다.
        """,
        metadata={"source": "vectordb_intro", "topic": "데이터베이스"}
    ),
    Document(
        page_content="""
        한국어 자연어 처리는 영어와 다른 특성을 가집니다. 한국어는 교착어로서
        조사와 어미가 단어에 붙어 문장의 의미를 결정합니다. 따라서 
        형태소 분석, 적절한 토큰화, 다국어 지원 임베딩 모델 사용이 중요합니다.
        KoNLPy, Mecab 등의 한국어 특화 도구를 활용할 수 있습니다.
        """,
        metadata={"source": "korean_nlp", "topic": "한국어"}
    ),
    Document(
        page_content="""
        프롬프트 엔지니어링은 LLM에게 효과적인 지시를 내리는 기술입니다.
        Zero-shot, Few-shot, Chain-of-Thought 등의 기법이 있습니다.
        좋은 프롬프트는 명확하고 구체적이며, 충분한 문맥을 제공해야 합니다.
        시스템 프롬프트를 통해 AI의 역할과 규칙을 정의할 수 있습니다.
        """,
        metadata={"source": "prompt_engineering", "topic": "프롬프트"}
    )
]            


test_texts = [
    '한국어 자연어 처리란 무엇인가요',
    'RAG 시스템의 장점을 설명해 주세요',
    '벡터 데이터베이스의 종류'
]


# 임베딩 모델 테스트 합치기
embedding_models = (KoreanEmbeddingModels.get_openai(),
    KoreanEmbeddingModels.get_bge_m3(),
    KoreanEmbeddingModels.get_multilingual_e5(),
    KoreanEmbeddingModels.get_korean_roberta())

embedding_model_names = ('openai', 'BGM-M3', 'Multilingul-E5', 'ko-RoBERT')
                         
def evaluate_embedding_models(embeddingmodel:KoreanEmbeddingModels, test_texts:List[str], model_name:str):
    start_time = time()
    vectors = embeddingmodel.embed_documents(test_texts)
    elapsed = time() - start_time
    print(f'{model_name} 임베딩==================')
    print(f'벡터 차원 : {len(vectors[0])}')
    print(f'처리시간 : {elapsed:.2f}')


# 각 임베딩 모델 평가
for idx, model in enumerate(embedding_models):
    evaluate_embedding_models(model, test_texts, embedding_model_names[idx])




# # 임베딩 모델 테스트 ---------------------> 개별
# # openai 임베딩 (base) 테스트
# openai_embeddings = KoreanEmbeddingModels.get_openai()  # @staticmethod로 만들어서 이렇게 사용가능?
# openai_start_time = time()
# openai_vectors = openai_embeddings.embed_documents(test_texts)
# openai_elapsed = time() - openai_start_time
# print('openai 임베딩==================')
# print(f'벡터 차원 : {len(openai_vectors)}')
# print(f'처리시간 : {openai_elapsed:.2f}')


# # BGE-M3 모델 테스트
# bge_m3_embeddings = KoreanEmbeddingModels.get_bge_m3()
# bge_m3_start_time = time()
# bge_m3_vectors = bge_m3_embeddings.embed_documents(test_texts)
# bge_m3_elapsed = time() - bge_m3_start_time
# print('BGE-M3 모델 임베딩==================')
# print(f'벡터 차원 : {len(bge_m3_vectors)}')
# print(f'처리시간 : {bge_m3_elapsed:.2f}')

# # Multilingual-E5 모델 테스트
# multi_e5_embeddings = KoreanEmbeddingModels.get_multilingual_e5()
# multi_e5_start_time = time()
# multi_e5_vectors = multi_e5_embeddings.embed_documents(test_texts)
# multi_e5_elapsed = time() - multi_e5_start_time
# print('Multilingual-E5 모델 임베딩==================')
# print(f'벡터 차원 : {len(multi_e5_vectors)}')
# print(f'처리시간 : {multi_e5_elapsed:.2f}')

# # BM-K/KoSimCSE-roberta-multitask 모델 테스트
# kosim_roberta_embeddings = KoreanEmbeddingModels.get_korean_roberta()
# kosim_roberta_start_time = time()
# kosim_roberta_vectors = kosim_roberta_embeddings.embed_documents(test_texts)
# kosim_roberta_elapsed = time() - kosim_roberta_start_time
# print('BM-K/KoSimCSE-roberta-multitask 모델 임베딩==================')
# print(f'벡터 차원 : {len(kosim_roberta_vectors)}')
# print(f'처리시간 : {kosim_roberta_elapsed:.2f}')



# vectorDB 구축 , 검색테스트
# 청킹(텍스트 분할)
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
doc_chunks = text_spliter.split_documents(korean_documents)
print(f'문서분할 완료 : {len(doc_chunks)}개 청크')


# 각 모델별 VectorDB 구축 및 테스트
for idx, model in enumerate(embedding_models): 
    # vectorDB 구축
    vectorstore = Chroma.from_documents(
        documents=doc_chunks,
        collection_name=f'korean_docs_{idx}',
        embedding=model
    )

    print(f'\n\n ======= {embedding_model_names[idx]}검색 테스트 결과 ===========')
    for query in test_texts:
        results = vectorstore.similarity_search(query)
        print(f'----->\n질문: {query}')
        print(f'검색결과: {results[0].metadata.get('topic', 'N/A')}')
        print(f'찾은 문장: {results[0].page_content}')



# 하이브리드 검색 
    # - 위에서 한국어 임베딩 중에 성능이 가장 좋은 모델을 선택해서 sparse vector 방식과 하이브리드로 연결(BM25)
    # RRF로 rank 형태로 최종 결과를 도출하던지, 또는 BM25 리트리버와 앙상블리트리버로 최종 구현(두 방식에 대한 가중치를 조정)
# 적절한 프롬프트와 llm을 체인방식으로 결합해서 결과를 도출
# 또는 랭그래프형태로 구현해도 됨


No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


openai 임베딩==================
벡터 차원 : 1536
처리시간 : 0.35
BGM-M3 임베딩==================
벡터 차원 : 1024
처리시간 : 0.36
Multilingul-E5 임베딩==================
벡터 차원 : 1024
처리시간 : 0.35
ko-RoBERT 임베딩==================
벡터 차원 : 768
처리시간 : 0.12
문서분할 완료 : 6개 청크


 ======= openai검색 테스트 결과 ===========
----->
질문: 한국어 자연어 처리란 무엇인가요
검색결과: 한국어
찾은 문장: 한국어 자연어 처리는 영어와 다른 특성을 가집니다. 한국어는 교착어로서
        조사와 어미가 단어에 붙어 문장의 의미를 결정합니다. 따라서 
        형태소 분석, 적절한 토큰화, 다국어 지원 임베딩 모델 사용이 중요합니다.
        KoNLPy, Mecab 등의 한국어 특화 도구를 활용할 수 있습니다.
----->
질문: RAG 시스템의 장점을 설명해 주세요
검색결과: RAG
찾은 문장: RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술로,
        LLM의 한계를 보완합니다. 기업의 내부 문서나 최신 정보를 벡터 
        데이터베이스에 저장하고, 사용자 질문과 관련된 문서를 검색하여
        답변의 정확성을 높입니다. 이를 통해 환각(Hallucination) 현상을 
        줄일 수 있습니다.
----->
질문: 벡터 데이터베이스의 종류
검색결과: 데이터베이스
찾은 문장: 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는
        데이터베이스입니다. 텍스트, 이미지, 오디오 등을 임베딩 벡터로 
        변환하여 저장하면, 의미적으로 유사한 항목을 빠르게 찾을 수 있습니다.
        ChromaDB, Pinecone, Weaviate, FAISS 등이 대표적입니다.


 =

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"